# ARC-v0.27 — NQ-GTE Independent Severity-Matched Boundary Confirmation

> **Syntax-audited fixed build.** Includes atomic/validated BEIR download, safe newline serialization, and corrected protocol SHA persistence.

## Purpose

Test whether the **representation-versus-search-effort approximation-feedback boundary** transfers to a new corpus and a new embedding-model family **after one-shot task-effectiveness severity is selected on FIT only and frozen before any validation trajectory outcome is generated**.

This experiment targets the main remaining external-validity gap in the current SIGIR full-paper line:

$$	ext{new corpus}+	ext{new encoder family}+	ext{FIT-only severity calibration}+	ext{untouched validation trajectories}.$$

### Frozen scientific design

- dataset: **BEIR Natural Questions (`nq`)**
- encoder: **`thenlper/gte-small`**
- representation contrast: **IVF-PQ32 → IVF-SQ8**, both `nprobe=64`
- search-effort high: **IVF-SQ8, `nprobe=64`**
- search-effort low: selected **on FIT only** from a frozen candidate grid
- same **44 anchored mean/softmax policies** as the current paper
- four feedback updates
- primary statistical unit: **query**
- primary estimand: `H3abs_representation - H3abs_matched_search_effort`

### Primary success criterion

The confirmatory mechanism contrast is supported iff the **paired query-bootstrap 95% CI** of the primary estimand lies strictly above zero. Representation H3abs itself is not required to be positive.

### Outcome-blindness boundary

This notebook makes **no global-pristine claim by default**. The intended claim is narrower: validation feedback-trajectory outcomes are not accessed before FIT calibration completes and the final confirmation protocol is frozen and hashed.

Negative, null, contracting, reversed, or failed-transfer outcomes are retained unchanged.

### Execution discipline

1. Run Cells 1–13.
2. Cell 13 performs FIT-only severity calibration.
3. Run Cell 14 to freeze the final confirmatory protocol and SHA-256.
4. Only after Cell 14 prints `CONFIRMATORY PROTOCOL FROZEN — PASS`, run Cells 15–20.
5. Run validation one-shot diagnostics only after the primary gate.
6. Do not alter the split, encoder, representation contrast, policy grid, candidate nprobe grid, endpoint definitions, or primary estimand after Cell 14.

In [ ]:
!pip -q install faiss-cpu sentence-transformers pyarrow scipy tqdm psutil

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib, json, math, os, random, time, warnings, zipfile

import faiss, numpy as np, pandas as pd, psutil, requests
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)
SEED=20260827
random.seed(SEED); np.random.seed(SEED)
DATASET_NAME='BEIR Natural Questions'; BEIR_NAME='nq'
ENCODER_NAME='thenlper/gte-small'; DIM=384
QUERY_PREFIX=''; PASSAGE_PREFIX=''
NLIST=4096; PQ_M=32; PQ_NBITS=8
REP_NPROBE=64; SEARCH_HIGH_NPROBE=64
SEARCH_LOW_CANDIDATES=[1,2,4,6,8,12,16,24,32]
TOP_RETRIEVE=100; UTILITY_K=10; MAX_ROUNDS=4
ALPHAS=[0.1,0.3,0.5,0.7]; MEAN_K=[5,20,50]; SOFTMAX_K=[5,20]; TEMPERATURES=[0.05,0.1,0.2,0.5]
BOOTSTRAP_REPS=10000; EPS_PRIMARY=0.002
SMOKE_N_QUERIES=10; CHECKPOINT_EVERY_QUERIES=10
PERSIST_LARGE_ARTIFACTS=False
ENCODE_BATCH=1024; CORPUS_BLOCK_ROWS=100000; TRAIN_SAMPLE=500000
faiss.omp_set_num_threads(os.cpu_count() or 1)
print('faiss:',faiss.__version__,'threads:',faiss.omp_get_max_threads(),'RAM GiB:',round(psutil.virtual_memory().total/2**30,2))

In [ ]:
DRIVE_ROOT=Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir(): drive.mount('/content/drive')
ARC_ROOT=DRIVE_ROOT/'rag-pq-checkpoints'/'arc-v0'; ARC_ROOT.mkdir(parents=True,exist_ok=True)
V027_ROOT=ARC_ROOT/'nq-gte-independent-severity-matched-v027'; V027_ROOT.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT=V027_ROOT/RUN_ID; OUT.mkdir(parents=True,exist_ok=False)
LOCAL_RAW_ROOT=Path('/content/arc-v027-nq-raw'); LOCAL_RAW_ROOT.mkdir(parents=True,exist_ok=True)
LARGE_ROOT=(DRIVE_ROOT/'rag-pq-checkpoints'/'arc-v027-nq-gte-large-cache') if PERSIST_LARGE_ARTIFACTS else Path('/content/arc-v027-nq-gte-large-cache')
LARGE_ROOT.mkdir(parents=True,exist_ok=True)
print('OUT:',OUT); print('LARGE_ROOT:',LARGE_ROOT)

In [ ]:
def sha256_file(path,chunk=16*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def membership_sha(ids): return hashlib.sha256('\n'.join(sorted(map(str,ids))).encode()).hexdigest()
def iter_jsonl(path):
    with open(path,encoding='utf-8') as f:
        for line in f:
            if line.strip(): yield json.loads(line)
def norm_vec(x,eps=1e-12):
    x=np.asarray(x,dtype=np.float32); return x/max(float(np.linalg.norm(x)),eps)
def slope(y):
    y=np.asarray(y,dtype=np.float64); x=np.arange(len(y),dtype=np.float64); return float(np.polyfit(x,y,1)[0])
def jacdist(a,b):
    A=set(map(int,a)); B=set(map(int,b)); return 1-len(A&B)/max(1,len(A|B))
def ndcg(ids,relevant,k=UTILITY_K):
    ids=np.asarray(ids,dtype=np.int64)[:k]; gains=np.asarray([1.0 if int(i) in relevant else 0.0 for i in ids]); discounts=1/np.log2(np.arange(2,len(ids)+2)); dcg=float(np.sum(gains*discounts)); m=min(k,len(relevant));
    return 0.0 if m==0 else dcg/float(np.sum(1/np.log2(np.arange(2,m+2))))
def mrr(ids,relevant,k=10):
    for rank,d in enumerate(np.asarray(ids)[:k],1):
        if int(d) in relevant: return 1.0/rank
    return 0.0
def recall(ids,relevant,k=10):
    return 0.0 if not relevant else len(set(map(int,np.asarray(ids)[:k]))&set(relevant))/len(relevant)
print('Helpers ready.')

## Phase A — Acquire BEIR NQ and create the outcome-blind split

The raw corpus is kept outside Drive by default. The split uses query IDs only, never retrieval outcomes.

In [ ]:
# Cell 4 — Robust download / resolve BEIR NQ locally
NQ_DIR = LOCAL_RAW_ROOT / "nq"
CORPUS_JSONL = NQ_DIR / "corpus.jsonl"
QUERIES_JSONL = NQ_DIR / "queries.jsonl"
ZIP_PATH = LOCAL_RAW_ROOT / "nq.zip"

NQ_URL = (
    "https://public.ukp.informatik.tu-darmstadt.de/"
    "thakur/BEIR/datasets/nq.zip"
)

def valid_zip(path):
    path = Path(path)
    if not path.is_file():
        return False
    try:
        if not zipfile.is_zipfile(path):
            return False
        with zipfile.ZipFile(path, "r") as zf:
            return zf.testzip() is None
    except Exception:
        return False

def download_zip_atomic(url, dst):
    dst = Path(dst)
    tmp = dst.with_suffix(dst.suffix + ".part")

    if tmp.exists():
        tmp.unlink()

    print("Downloading:", url)

    with requests.get(
        url,
        stream=True,
        timeout=(30, 300),
        allow_redirects=True,
    ) as r:
        print("HTTP status:", r.status_code)
        print("Content-Type:", r.headers.get("content-type"))
        print("Content-Length:", r.headers.get("content-length"))
        r.raise_for_status()

        total = int(r.headers.get("content-length", 0))

        with open(tmp, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="nq.zip",
        ) as bar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    print("Downloaded bytes:", f"{tmp.stat().st_size:,}")

    if not valid_zip(tmp):
        with open(tmp, "rb") as f:
            prefix = f.read(200)
        print("Invalid download prefix:", repr(prefix))
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            "Downloaded NQ artifact is not a valid ZIP archive. "
            "The server may have returned an HTML/error response."
        )

    tmp.replace(dst)
    print("ZIP validation — PASS")

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file()):
    if ZIP_PATH.exists() and not valid_zip(ZIP_PATH):
        print("Removing corrupt/incomplete archive:", ZIP_PATH)
        print("Corrupt archive bytes:", f"{ZIP_PATH.stat().st_size:,}")
        ZIP_PATH.unlink()

    if not ZIP_PATH.is_file():
        download_zip_atomic(NQ_URL, ZIP_PATH)

    assert valid_zip(ZIP_PATH), ZIP_PATH

    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_RAW_ROOT)
    print("Extraction — COMPLETE")

assert CORPUS_JSONL.is_file(), CORPUS_JSONL
assert QUERIES_JSONL.is_file(), QUERIES_JSONL

qrel_candidates = sorted((NQ_DIR / "qrels").glob("*.tsv"))
assert qrel_candidates, f"No qrels TSV found under {NQ_DIR / 'qrels'}"

QRELS_PATH = next(
    (p for p in qrel_candidates if p.name == "test.tsv"),
    qrel_candidates[0],
)

print("=" * 80)
print("BEIR NQ DATASET — READY")
print("=" * 80)
print("Corpus :", CORPUS_JSONL)
print("Queries:", QUERIES_JSONL)
print("Qrels  :", QRELS_PATH)
print("Corpus bytes :", f"{CORPUS_JSONL.stat().st_size:,}")
print("Queries bytes:", f"{QUERIES_JSONL.stat().st_size:,}")
print("Qrels bytes  :", f"{QRELS_PATH.stat().st_size:,}")
print("Cell 4 — PASS")

In [ ]:
qrels_df=pd.read_csv(QRELS_PATH,sep='\t')
q_qid_col=next(c for c in ['query-id','query_id','qid'] if c in qrels_df.columns)
q_doc_col=next(c for c in ['corpus-id','corpus_id','doc_id'] if c in qrels_df.columns)
q_score_col=next((c for c in ['score','relevance','rel'] if c in qrels_df.columns),None)
qrels_df[q_qid_col]=qrels_df[q_qid_col].astype(str); qrels_df[q_doc_col]=qrels_df[q_doc_col].astype(str)
if q_score_col is not None:
    qrels_df[q_score_col]=pd.to_numeric(qrels_df[q_score_col],errors='coerce'); qrels_df=qrels_df[qrels_df[q_score_col]>0].copy()
query_text={str(o['_id']):str(o.get('text','')) for o in iter_jsonl(QUERIES_JSONL)}
ALL_QUERY_IDS=sorted(set(qrels_df[q_qid_col])&set(query_text)); assert len(ALL_QUERY_IDS)>=1000
def split_key(qid): return hashlib.sha256(f'{SEED}|{qid}'.encode()).hexdigest()
ordered=sorted(ALL_QUERY_IDS,key=split_key); cut=len(ordered)//2; FIT_IDS=ordered[:cut]; VAL_IDS=ordered[cut:]
assert set(FIT_IDS).isdisjoint(VAL_IDS)
split_df=pd.DataFrame({'query_id':FIT_IDS+VAL_IDS,'split':['fit']*len(FIT_IDS)+['validation']*len(VAL_IDS)})
SPLIT_PATH=OUT/'v027_nq_query_split.csv'; split_df.to_csv(SPLIT_PATH,index=False)
print('N=',len(ALL_QUERY_IDS),'FIT=',len(FIT_IDS),'VAL=',len(VAL_IDS)); print('FIT SHA',membership_sha(FIT_IDS)); print('VAL SHA',membership_sha(VAL_IDS))

In [ ]:
# Cell 6 — Freeze FIT calibration design BEFORE any FIT retrieval outcomes
POLICIES = []
for alpha in ALPHAS:
    for k in MEAN_K:
        POLICIES.append({
            "method": "mean",
            "alpha": float(alpha),
            "k": int(k),
            "temperature": np.nan,
            "config_key": f"mean-k{k}-a{str(alpha).replace('.', 'p')}-tnone",
        })
    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "method": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": (
                    f"softmax-k{k}-a{str(alpha).replace('.', 'p')}-"
                    f"t{str(tau).replace('.', 'p')}"
                ),
            })

assert len(POLICIES) == 44

POLICY_PATH = OUT / "v027_frozen_policy_grid.csv"
pd.DataFrame(POLICIES).to_csv(POLICY_PATH, index=False)

CALIBRATION_PROTOCOL = {
    "status": "ARC_V027_FIT_CALIBRATION_DESIGN_FROZEN_BEFORE_FIT_OUTCOMES",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "dataset": DATASET_NAME,
    "encoder": ENCODER_NAME,
    "dimension": DIM,
    "global_pristine_claim": False,
    "validation_trajectory_outcome_blind_target": True,
    "split": {
        "n_total": len(ALL_QUERY_IDS),
        "n_fit": len(FIT_IDS),
        "n_validation": len(VAL_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "validation_sha256": membership_sha(VAL_IDS),
    },
    "representation": {
        "low": "IVF-PQ32",
        "high": "IVF-SQ8",
        "nprobe": REP_NPROBE,
        "nlist": NLIST,
        "pq_m": PQ_M,
        "pq_nbits": PQ_NBITS,
    },
    "search_effort": {
        "representation": "same IVF-SQ8",
        "high_nprobe": SEARCH_HIGH_NPROBE,
        "candidate_low_nprobes": SEARCH_LOW_CANDIDATES,
        "selection_data": "FIT only",
        "selection_metric": (
            "absolute mismatch between mean one-shot nDCG@10 "
            "high-minus-low gap and representation high-minus-low gap"
        ),
        "tie_breaker": "smaller nprobe",
    },
    "feedback": {
        "policy_count": len(POLICIES),
        "rounds": MAX_ROUNDS,
        "top_retrieve": TOP_RETRIEVE,
        "utility_k": UTILITY_K,
    },
    "primary_confirmatory_estimand": (
        "query-averaged H3abs representation minus "
        "severity-matched search effort"
    ),
    "primary_success_criterion": (
        "paired query-bootstrap 95% CI strictly above zero"
    ),
    "representation_sign_is_required": False,
    "negative_null_reversal_retained": True,
    "validation_trajectory_retuning_allowed": False,
}

CALIBRATION_PROTOCOL_PATH = OUT / "v027_fit_calibration_protocol.json"
CALIBRATION_PROTOCOL_PATH.write_text(
    json.dumps(CALIBRATION_PROTOCOL, indent=2, sort_keys=True),
    encoding="utf-8",
)

CALIBRATION_PROTOCOL_SHA = sha256_file(CALIBRATION_PROTOCOL_PATH)

CALIBRATION_SHA_PATH = OUT / "V027_FIT_CALIBRATION_PROTOCOL_SHA256.txt"
CALIBRATION_SHA_PATH.write_text(
    (
        f"{CALIBRATION_PROTOCOL_SHA}  "
        f"{CALIBRATION_PROTOCOL_PATH.name}\n"
    ),
    encoding="utf-8",
)

print("FIT CALIBRATION DESIGN FROZEN — PASS")
print("Calibration protocol SHA:", CALIBRATION_PROTOCOL_SHA)

## Phase B — Encode GTE queries and the full NQ corpus

The full corpus is encoded in resumable float16 blocks. Large embeddings/indexes remain ephemeral by default.

In [ ]:
# Cell 7 — Encode all evaluation queries
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(ENCODER_NAME, device=device)

QUERY_IDS_PATH = LARGE_ROOT / "nq_query_ids.txt"
QUERY_EMB_PATH = LARGE_ROOT / "nq_query_embeddings.float32.npy"

if QUERY_EMB_PATH.is_file() and QUERY_IDS_PATH.is_file():
    query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode="r")
    assert QUERY_IDS_PATH.read_text(encoding="utf-8").splitlines() == ALL_QUERY_IDS
else:
    query_embeddings = model.encode(
        [QUERY_PREFIX + query_text[qid] for qid in ALL_QUERY_IDS],
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    np.save(QUERY_EMB_PATH, query_embeddings)
    QUERY_IDS_PATH.write_text(
        "\n".join(ALL_QUERY_IDS) + "\n",
        encoding="utf-8",
    )

QUERY_INDEX = {qid: i for i, qid in enumerate(ALL_QUERY_IDS)}

assert query_embeddings.shape == (len(ALL_QUERY_IDS), DIM)
assert np.isfinite(np.asarray(query_embeddings[: min(100, len(query_embeddings))])).all()

print("Query embeddings:", query_embeddings.shape)
print("Device:", device)
print("QUERY ALIGNMENT — PASS")

In [ ]:
BLOCK_DIR=LARGE_ROOT/'corpus_blocks'; BLOCK_DIR.mkdir(parents=True,exist_ok=True)
RELEVANT_DOC_IDS=set(qrels_df[q_doc_col].astype(str)); QREL_ROW_MAP_PATH=LARGE_ROOT/'nq_qrel_doc_rows.csv'; CORPUS_MANIFEST_PATH=LARGE_ROOT/'nq_corpus_block_manifest.json'
block_records=[]; relevant_row_map={}; texts_buf=[]; block_idx=0; global_row=0
def save_or_reuse_block(idx,texts,row_start):
    ep=BLOCK_DIR/f'block-{idx:04d}.float16.npy'
    if ep.is_file():
        arr=np.load(ep,mmap_mode='r'); assert arr.shape==(len(texts),DIM); return {'block':idx,'row_start':row_start,'row_end':row_start+len(texts),'rows':len(texts),'path':str(ep),'sha256':sha256_file(ep),'reused':True}
    vec=model.encode([PASSAGE_PREFIX+t for t in texts],batch_size=ENCODE_BATCH,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True).astype(np.float16); np.save(ep,vec); return {'block':idx,'row_start':row_start,'row_end':row_start+len(texts),'rows':len(texts),'path':str(ep),'sha256':sha256_file(ep),'reused':False}
for obj in iter_jsonl(CORPUS_JSONL):
    doc_id=str(obj['_id']); title=str(obj.get('title','')); text=str(obj.get('text','')); full=(title+' '+text).strip() if title else text
    if doc_id in RELEVANT_DOC_IDS: relevant_row_map[doc_id]=global_row
    texts_buf.append(full); global_row+=1
    if len(texts_buf)>=CORPUS_BLOCK_ROWS:
        row_start=global_row-len(texts_buf); block_records.append(save_or_reuse_block(block_idx,texts_buf,row_start)); texts_buf=[]; block_idx+=1
if texts_buf:
    row_start=global_row-len(texts_buf); block_records.append(save_or_reuse_block(block_idx,texts_buf,row_start))
N_DOCS=global_row; assert sum(r['rows'] for r in block_records)==N_DOCS; assert not (RELEVANT_DOC_IDS-set(relevant_row_map))
pd.DataFrame([{'doc_id':d,'corpus_row':r} for d,r in relevant_row_map.items()]).to_csv(QREL_ROW_MAP_PATH,index=False)
CORPUS_MANIFEST_PATH.write_text(json.dumps({'status':'COMPLETE','encoder':ENCODER_NAME,'dimension':DIM,'corpus_rows':N_DOCS,'blocks':block_records,'qrels_relevant_docs_mapped':len(relevant_row_map)},indent=2)); print('Corpus rows',f'{N_DOCS:,}','blocks',len(block_records))

In [ ]:
blocks=[]; block_starts=[]; block_ends=[]
for r in block_records:
    arr=np.load(r['path'],mmap_mode='r'); blocks.append(arr); block_starts.append(int(r['row_start'])); block_ends.append(int(r['row_end']))
block_starts=np.asarray(block_starts,dtype=np.int64); block_ends=np.asarray(block_ends,dtype=np.int64)
def fetch_docs(rows):
    rows=np.asarray(rows,dtype=np.int64); out=np.empty((len(rows),DIM),dtype=np.float32); bidx=np.searchsorted(block_ends,rows,side='right')
    for b in np.unique(bidx):
        mask=bidx==b; out[mask]=np.asarray(blocks[b][rows[mask]-block_starts[b]],dtype=np.float32)
    out/=np.maximum(np.linalg.norm(out,axis=1,keepdims=True),1e-12); return out
doc_to_row={str(k):int(v) for k,v in relevant_row_map.items()}; QRELS=defaultdict(set)
for _,r in qrels_df.iterrows():
    qid=str(r[q_qid_col]); docid=str(r[q_doc_col]);
    if qid in QUERY_INDEX and docid in doc_to_row: QRELS[qid].add(doc_to_row[docid])
assert all(QRELS[q] for q in ALL_QUERY_IDS); print('Qrels row-space alignment — PASS')

In [ ]:
TRAIN_PATH=LARGE_ROOT/f'train_sample_{TRAIN_SAMPLE}.float32.npy'
if TRAIN_PATH.is_file(): train=np.load(TRAIN_PATH,mmap_mode='r')
else:
    rng=np.random.default_rng(SEED+270); per=int(math.ceil(TRAIN_SAMPLE/len(blocks))); pieces=[]
    for arr in blocks:
        n=len(arr); take=min(per,n); idx=rng.choice(n,size=take,replace=False); pieces.append(np.asarray(arr[idx],dtype=np.float32))
    train=np.concatenate(pieces,axis=0)[:TRAIN_SAMPLE]; train/=np.maximum(np.linalg.norm(train,axis=1,keepdims=True),1e-12); np.save(TRAIN_PATH,train)
print('Training sample',train.shape)

In [ ]:
PQ_PATH=LARGE_ROOT/'nq-gte-ivfpq-nlist4096-m32-nbits8.faiss'; SQ_PATH=LARGE_ROOT/'nq-gte-ivfsq8-nlist4096.faiss'
def stream_add(index):
    added=0
    for arr in tqdm(blocks,desc='adding corpus blocks'):
        x=np.asarray(arr,dtype=np.float32); x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12); index.add(x); added+=len(x)
    return added
if PQ_PATH.is_file(): pq32=faiss.read_index(str(PQ_PATH))
else:
    q=faiss.IndexFlatIP(DIM); pq32=faiss.IndexIVFPQ(q,DIM,NLIST,PQ_M,PQ_NBITS,faiss.METRIC_INNER_PRODUCT); pq32.train(np.asarray(train)); assert stream_add(pq32)==N_DOCS; faiss.write_index(pq32,str(PQ_PATH))
if SQ_PATH.is_file(): sq8=faiss.read_index(str(SQ_PATH))
else:
    q=faiss.IndexFlatIP(DIM); sq8=faiss.IndexIVFScalarQuantizer(q,DIM,NLIST,faiss.ScalarQuantizer.QT_8bit,faiss.METRIC_INNER_PRODUCT); sq8.train(np.asarray(train)); assert stream_add(sq8)==N_DOCS; faiss.write_index(sq8,str(SQ_PATH))
assert pq32.ntotal==sq8.ntotal==N_DOCS; print('INDEX BUILD/LOAD — PASS',f'{N_DOCS:,}')

In [ ]:
def search_index(index,q,nprobe):
    index.nprobe=int(nprobe); s,I=index.search(norm_vec(q)[None,:],TOP_RETRIEVE); valid=I[0]>=0; return s[0][valid],I[0][valid]
def one_shot_query(qid):
    q=query_embeddings[QUERY_INDEX[qid]]; rel=QRELS[qid]; _,i_pq=search_index(pq32,q,REP_NPROBE); _,i_hi=search_index(sq8,q,SEARCH_HIGH_NPROBE)
    row={'query_id':qid,'pq32_n64_ndcg10':ndcg(i_pq,rel),'sq8_n64_ndcg10':ndcg(i_hi,rel),'pq32_n64_jaccard_vs_high':jacdist(i_pq,i_hi)}
    for np_low in SEARCH_LOW_CANDIDATES:
        _,i_low=search_index(sq8,q,np_low); row[f'sq8_n{np_low}_ndcg10']=ndcg(i_low,rel); row[f'sq8_n{np_low}_jaccard_vs_high']=jacdist(i_low,i_hi)
    return row
print('One-shot helpers ready.')

## Phase C — FIT-only severity calibration

Only this stage may select the low search-effort condition. Candidate-set Jaccard mismatch is reported as a secondary calibration diagnostic but does not drive retuning.

In [ ]:
FIT_ONE_SHOT_PATH=OUT/'v027_fit_one_shot_severity_table.csv'; fit_one=pd.DataFrame([one_shot_query(q) for q in tqdm(FIT_IDS,desc='FIT one-shot calibration')]); fit_one.to_csv(FIT_ONE_SHOT_PATH,index=False)
rep_gap=float((fit_one['sq8_n64_ndcg10']-fit_one['pq32_n64_ndcg10']).mean()); rep_jac=float(fit_one['pq32_n64_jaccard_vs_high'].mean()); rows=[]
for np_low in SEARCH_LOW_CANDIDATES:
    search_gap=float((fit_one['sq8_n64_ndcg10']-fit_one[f'sq8_n{np_low}_ndcg10']).mean()); search_jac=float(fit_one[f'sq8_n{np_low}_jaccard_vs_high'].mean()); abs_m=abs(search_gap-rep_gap); rows.append({'nprobe_low':np_low,'representation_gap_ndcg10':rep_gap,'search_gap_ndcg10':search_gap,'absolute_gap_mismatch':abs_m,'relative_gap_mismatch':abs_m/max(abs(rep_gap),1e-12),'representation_jaccard_vs_high':rep_jac,'search_jaccard_vs_high':search_jac,'absolute_jaccard_mismatch':abs(search_jac-rep_jac)})
match_df=pd.DataFrame(rows).sort_values(['absolute_gap_mismatch','nprobe_low']).reset_index(drop=True); MATCHED_NPROBE=int(match_df.iloc[0].nprobe_low); MATCHED_SEARCH_GAP=float(match_df.iloc[0].search_gap_ndcg10); MATCHED_REL_MISMATCH=float(match_df.iloc[0].relative_gap_mismatch); MATCH_TABLE_PATH=OUT/'v027_fit_severity_match_table.csv'; match_df.to_csv(MATCH_TABLE_PATH,index=False); display(match_df.round(6)); print('MATCHED_NPROBE=',MATCHED_NPROBE,'rel mismatch=',MATCHED_REL_MISMATCH); print('DO NOT RETUNE AFTER THIS CELL.')

In [ ]:
# Cell 14 — Freeze FINAL confirmatory protocol BEFORE validation trajectories
CONFIRM_PROTOCOL = {
    "status": (
        "ARC_V027_NQ_GTE_CONFIRMATORY_PROTOCOL_"
        "FROZEN_BEFORE_VALIDATION_TRAJECTORIES"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "dataset": DATASET_NAME,
    "encoder": ENCODER_NAME,
    "dimension": DIM,
    "global_pristine_claim": False,
    "validation_trajectory_outcome_blind_claim": (
        "No validation feedback-trajectory outcome was generated or inspected "
        "before this final protocol was frozen."
    ),
    "calibration_protocol_sha256": CALIBRATION_PROTOCOL_SHA,
    "split": {
        "n_fit": len(FIT_IDS),
        "n_validation": len(VAL_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "validation_sha256": membership_sha(VAL_IDS),
    },
    "fit_severity_calibration": {
        "metric": "mean one-shot nDCG@10 high-minus-low gap",
        "representation_gap": rep_gap,
        "candidate_low_nprobes": SEARCH_LOW_CANDIDATES,
        "selected_nprobe_low": MATCHED_NPROBE,
        "selected_search_gap": MATCHED_SEARCH_GAP,
        "relative_gap_mismatch": MATCHED_REL_MISMATCH,
        "candidate_jaccard_is_secondary_diagnostic_only": True,
        "post_selection_retuning_allowed": False,
    },
    "retrievers": {
        "representation_low": {
            "type": "IndexIVFPQ",
            "nlist": NLIST,
            "m": PQ_M,
            "nbits": PQ_NBITS,
            "nprobe": REP_NPROBE,
        },
        "representation_high": {
            "type": "IndexIVFScalarQuantizer",
            "quantizer": "QT_8bit",
            "nlist": NLIST,
            "nprobe": REP_NPROBE,
        },
        "search_effort_low": {
            "type": "same IVF-SQ8",
            "nprobe": MATCHED_NPROBE,
        },
        "search_effort_high": {
            "type": "same IVF-SQ8",
            "nprobe": SEARCH_HIGH_NPROBE,
        },
    },
    "feedback": {
        "policy_count": len(POLICIES),
        "rounds": MAX_ROUNDS,
        "top_retrieve": TOP_RETRIEVE,
        "utility_k": UTILITY_K,
        "alphas": ALPHAS,
        "mean_k": MEAN_K,
        "softmax_k": SOFTMAX_K,
        "temperatures": TEMPERATURES,
        "anchoring": "q_(t+1)=normalize((1-alpha)q0 + alpha*F_t)",
    },
    "primary_confirmatory_estimand": (
        "query-averaged H3_abs_slope representation minus "
        "query-averaged H3_abs_slope severity-matched search effort"
    ),
    "primary_success_criterion": (
        "10,000-replicate paired query-bootstrap 95% CI strictly above zero"
    ),
    "representation_positive_is_not_required": True,
    "negative_null_or_reversal_results_retained": True,
    "validation_retuning_allowed": False,
    "validation_one_shot_deferred_until_after_primary_gate": True,
}

CONFIRM_PROTOCOL_PATH = OUT / "v027_frozen_confirmation_protocol.json"
CONFIRM_PROTOCOL_PATH.write_text(
    json.dumps(CONFIRM_PROTOCOL, indent=2, sort_keys=True),
    encoding="utf-8",
)

CONFIRM_PROTOCOL_SHA = sha256_file(CONFIRM_PROTOCOL_PATH)

CONFIRM_SHA_PATH = OUT / "V027_CONFIRMATION_PROTOCOL_SHA256.txt"
CONFIRM_SHA_PATH.write_text(
    f"{CONFIRM_PROTOCOL_SHA}  {CONFIRM_PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("CONFIRMATORY PROTOCOL FROZEN — PASS")
print("Confirmation protocol SHA:", CONFIRM_PROTOCOL_SHA)
print("MATCHED_NPROBE:", MATCHED_NPROBE)
print("Do not alter the match or validation design after this point.")

## Phase D — Confirmatory validation trajectories

Do not run until the final confirmation protocol is frozen.

In [ ]:
MECHANISMS=['representation','search_effort']
def retrieve(mechanism,fidelity,q):
    if mechanism=='representation': return search_index(pq32 if fidelity=='L' else sq8,q,REP_NPROBE)
    if mechanism=='search_effort': return search_index(sq8,q,MATCHED_NPROBE if fidelity=='L' else SEARCH_HIGH_NPROBE)
    raise ValueError(mechanism)
def feedback_vector(scores,ids,policy):
    k=int(policy['k']); ids=np.asarray(ids[:k],dtype=np.int64); scores=np.asarray(scores[:k],dtype=np.float64); docs=fetch_docs(ids)
    if policy['method']=='mean': f=docs.mean(axis=0)
    else:
        z=scores/float(policy['temperature']); z-=z.max(); w=np.exp(z); w/=w.sum(); f=(docs*w[:,None]).sum(axis=0)
    return norm_vec(f)
def update_state(q0,f,alpha): return norm_vec((1-float(alpha))*q0+float(alpha)*f)
def run_pair(qid,policy,mechanism):
    q0=norm_vec(query_embeddings[QUERY_INDEX[qid]]); qL=q0.copy(); qH=q0.copy(); rel=QRELS[qid]; rows=[]
    for t in range(MAX_ROUNDS+1):
        sL,iL=retrieve(mechanism,'L',qL); sH,iH=retrieve(mechanism,'H',qH); uL=ndcg(iL,rel); uH=ndcg(iH,rel)
        rows.append({'query_id':str(qid),'mechanism':mechanism,'iteration':t,'method':policy['method'],'alpha':float(policy['alpha']),'k':int(policy['k']),'temperature':float(policy['temperature']) if not pd.isna(policy['temperature']) else np.nan,'config_key':policy['config_key'],'query_state_distance':1-float(np.dot(norm_vec(qL),norm_vec(qH))),'candidate_jaccard_distance':jacdist(iL[:TOP_RETRIEVE],iH[:TOP_RETRIEVE]),'utility_low':uL,'utility_high':uH,'signed_utility_gap':uH-uL,'abs_utility_gap':abs(uH-uL)})
        if t==MAX_ROUNDS: break
        qL=update_state(q0,feedback_vector(sL,iL,policy),policy['alpha']); qH=update_state(q0,feedback_vector(sH,iH,policy),policy['alpha'])
    return rows
def endpoint_df(traj):
    out=[]; cols=['query_id','mechanism','method','alpha','k','temperature','config_key']
    for keys,g in traj.groupby(cols,dropna=False,sort=False):
        g=g.sort_values('iteration'); out.append({'query_id':keys[0],'mechanism':keys[1],'method':keys[2],'alpha':keys[3],'k':keys[4],'temperature':keys[5],'config_key':keys[6],'H1_slope':slope(g.query_state_distance),'H2_slope':slope(g.candidate_jaccard_distance),'H3_abs_slope':slope(g.abs_utility_gap),'H3_signed_slope':slope(g.signed_utility_gap),'final_signed_gap':float(g.signed_utility_gap.iloc[-1])})
    return pd.DataFrame(out)
print('Trajectory helpers ready.')

In [ ]:
smoke_qids=FIT_IDS[:min(SMOKE_N_QUERIES,len(FIT_IDS))]; assert set(smoke_qids).isdisjoint(VAL_IDS)
for mech in MECHANISMS:
    for policy in [POLICIES[0],POLICIES[-1]]:
        s=pd.DataFrame(run_pair(smoke_qids[0],policy,mech)); assert len(s)==MAX_ROUNDS+1; assert np.isfinite(s[['query_state_distance','candidate_jaccard_distance','utility_low','utility_high','signed_utility_gap','abs_utility_gap']].to_numpy()).all()
print('FIT-ONLY IMPLEMENTATION SMOKE — PASS')

In [ ]:
RUN_DIR=OUT/'validation-confirmatory'; RUN_DIR.mkdir(parents=True,exist_ok=True); expected_rows=len(VAL_IDS)*len(MECHANISMS)*len(POLICIES)*(MAX_ROUNDS+1); print('Expected rows',f'{expected_rows:,}')
for start in range(0,len(VAL_IDS),CHECKPOINT_EVERY_QUERIES):
    stop=min(start+CHECKPOINT_EVERY_QUERIES,len(VAL_IDS)); cp=RUN_DIR/f'traj_{start:05d}_{stop:05d}.parquet'
    if cp.exists(): print('skip',cp.name); continue
    rows=[]; t0=time.perf_counter()
    for qi in range(start,stop):
        qid=VAL_IDS[qi]
        for mech in MECHANISMS:
            for policy in POLICIES: rows.extend(run_pair(qid,policy,mech))
    df=pd.DataFrame(rows); tmp=cp.with_suffix('.tmp.parquet'); df.to_parquet(tmp,index=False); os.replace(tmp,cp); print('wrote',cp.name,f'{len(df):,}','rows',f'{time.perf_counter()-t0:.1f}s')
parts=sorted(RUN_DIR.glob('traj_*.parquet')); traj=pd.concat([pd.read_parquet(p) for p in parts],ignore_index=True); assert len(traj)==expected_rows; assert traj.query_id.nunique()==len(VAL_IDS); TRAJ_PATH=OUT/'v027_validation_confirmatory_trajectories.parquet'; traj.to_parquet(TRAJ_PATH,index=False); endpoints=endpoint_df(traj); ENDPOINT_PATH=OUT/'v027_validation_confirmatory_endpoints.parquet'; endpoints.to_parquet(ENDPOINT_PATH,index=False); print('CONFIRMATORY VALIDATION SWEEP — COMPLETE',f'{len(traj):,}',f'{len(endpoints):,}')

In [ ]:
MEASURES=['H1_slope','H2_slope','H3_abs_slope','H3_signed_slope']; q=endpoints.groupby(['query_id','mechanism'],as_index=False)[MEASURES].mean(); rep=q[q.mechanism.eq('representation')].set_index('query_id'); sea=q[q.mechanism.eq('search_effort')].set_index('query_id'); rep_h3=rep.loc[VAL_IDS,'H3_abs_slope'].to_numpy(float); sea_h3=sea.loc[VAL_IDS,'H3_abs_slope'].to_numpy(float); paired=rep_h3-sea_h3; rng=np.random.default_rng(SEED+2701)
def boot(x,reps=BOOTSTRAP_REPS):
    x=np.asarray(x,float); n=len(x); b=np.empty(reps)
    for i in range(reps): b[i]=x[rng.integers(0,n,size=n)].mean()
    lo,hi=np.quantile(b,[.025,.975]); return float(x.mean()),float(lo),float(hi),n
rep_stat=boot(rep_h3); sea_stat=boot(sea_h3); diff_stat=boot(paired); primary=pd.DataFrame([{'estimand':'representation_H3abs','mean':rep_stat[0],'ci95_low':rep_stat[1],'ci95_high':rep_stat[2],'n_queries':rep_stat[3]},{'estimand':'severity_matched_search_H3abs','mean':sea_stat[0],'ci95_low':sea_stat[1],'ci95_high':sea_stat[2],'n_queries':sea_stat[3]},{'estimand':'representation_minus_search_H3abs','mean':diff_stat[0],'ci95_low':diff_stat[1],'ci95_high':diff_stat[2],'n_queries':diff_stat[3]}]); PRIMARY_PATH=OUT/'v027_primary_paired_query_bootstrap.csv'; primary.to_csv(PRIMARY_PATH,index=False); display(primary.round(6)); PRIMARY_SUPPORTED=bool(diff_stat[1]>0); print('PRIMARY SUPPORTED:',PRIMARY_SUPPORTED)

In [ ]:
fam=endpoints.groupby(['query_id','mechanism','method'],as_index=False)['H3_abs_slope'].mean(); rows=[]
for family in ['mean','softmax']:
    r=fam[(fam.mechanism=='representation')&(fam.method==family)].set_index('query_id').loc[VAL_IDS,'H3_abs_slope'].to_numpy(); s=fam[(fam.mechanism=='search_effort')&(fam.method==family)].set_index('query_id').loc[VAL_IDS,'H3_abs_slope'].to_numpy(); st=boot(r-s); rows.append({'estimand':f'{family}_representation_minus_search_H3abs','mean':st[0],'ci95_low':st[1],'ci95_high':st[2],'n_queries':st[3]})
piv=fam.pivot_table(index=['query_id','mechanism'],columns='method',values='H3_abs_slope'); piv['balanced']=.5*piv['mean']+.5*piv['softmax']; rb=piv.xs('representation',level='mechanism').loc[VAL_IDS,'balanced'].to_numpy(); sb=piv.xs('search_effort',level='mechanism').loc[VAL_IDS,'balanced'].to_numpy(); st=boot(rb-sb); rows.append({'estimand':'equal_family_representation_minus_search_H3abs','mean':st[0],'ci95_low':st[1],'ci95_high':st[2],'n_queries':st[3]}); family_summary=pd.DataFrame(rows); FAMILY_PATH=OUT/'v027_secondary_family_robustness.csv'; family_summary.to_csv(FAMILY_PATH,index=False); display(family_summary.round(6))

In [ ]:
gate={'status':'ARC_V027_NQ_GTE_INDEPENDENT_SEVERITY_MATCHED_CONFIRMATION_ANALYZED','protocol_sha256':CONFIRM_PROTOCOL_SHA,'dataset':DATASET_NAME,'encoder':ENCODER_NAME,'n_validation_queries':len(VAL_IDS),'matched_nprobe_low':MATCHED_NPROBE,'fit_representation_gap_ndcg10':rep_gap,'fit_matched_search_gap_ndcg10':MATCHED_SEARCH_GAP,'fit_relative_gap_mismatch':MATCHED_REL_MISMATCH,'representation_H3abs':{'mean':rep_stat[0],'ci95':[rep_stat[1],rep_stat[2]]},'severity_matched_search_H3abs':{'mean':sea_stat[0],'ci95':[sea_stat[1],sea_stat[2]]},'paired_representation_minus_search_H3abs':{'mean':diff_stat[0],'ci95':[diff_stat[1],diff_stat[2]],'ci_excludes_zero_positive':PRIMARY_SUPPORTED},'primary_confirmatory_boundary_supported':PRIMARY_SUPPORTED,'negative_null_or_reversal_retained':True,'validation_retuning_performed':False,'validation_one_shot_accessed_before_primary_gate':False,'completed_at_utc':datetime.now(timezone.utc).isoformat()}; GATE_PATH=OUT/'v027_primary_confirmation_gate.json'; GATE_PATH.write_text(json.dumps(gate,indent=2,sort_keys=True)); print(json.dumps(gate,indent=2))

## Phase E — Post-primary validation diagnostics

Run only after the primary gate is persisted. These diagnostics do not redefine the primary claim.

In [ ]:
assert GATE_PATH.is_file(); rows=[]
for qid in tqdm(VAL_IDS,desc='post-primary validation one-shot'):
    qv=query_embeddings[QUERY_INDEX[qid]]; rel=QRELS[qid]; _,a=search_index(pq32,qv,REP_NPROBE); _,h=search_index(sq8,qv,SEARCH_HIGH_NPROBE); _,s=search_index(sq8,qv,MATCHED_NPROBE); rows.append({'query_id':qid,'rep_low_ndcg10':ndcg(a,rel),'search_low_ndcg10':ndcg(s,rel),'high_ndcg10':ndcg(h,rel),'rep_low_mrr10':mrr(a,rel),'search_low_mrr10':mrr(s,rel),'high_mrr10':mrr(h,rel),'rep_low_recall10':recall(a,rel,10),'search_low_recall10':recall(s,rel,10),'high_recall10':recall(h,rel,10),'rep_low_recall100':recall(a,rel,100),'search_low_recall100':recall(s,rel,100),'high_recall100':recall(h,rel,100),'rep_jaccard_vs_high':jacdist(a,h),'search_jaccard_vs_high':jacdist(s,h)})
val_one=pd.DataFrame(rows); VAL_ONE_PATH=OUT/'v027_postprimary_validation_one_shot_multimetric.csv'; val_one.to_csv(VAL_ONE_PATH,index=False); display(val_one.mean(numeric_only=True).to_frame('mean').round(6))

In [ ]:
reg=[]
for mech in MECHANISMS:
    h=endpoints.loc[endpoints.mechanism.eq(mech),'H3_abs_slope'].to_numpy(float); reg.append({'mechanism':mech,'epsilon':EPS_PRIMARY,'stable_null_fraction':float((np.abs(h)<=EPS_PRIMARY).mean()),'amplifying_fraction':float((h>EPS_PRIMARY).mean()),'contracting_fraction':float((h<-EPS_PRIMARY).mean())})
regimes=pd.DataFrame(reg); REGIME_PATH=OUT/'v027_regime_summary.csv'; regimes.to_csv(REGIME_PATH,index=False); display(regimes.round(6))

In [ ]:
REPORT={**gate,'calibration_protocol_sha256':CALIBRATION_PROTOCOL_SHA,'confirmation_protocol_sha256':CONFIRM_PROTOCOL_SHA,'split_path':str(SPLIT_PATH),'fit_one_shot_path':str(FIT_ONE_SHOT_PATH),'fit_match_table_path':str(MATCH_TABLE_PATH),'trajectory_path':str(TRAJ_PATH),'endpoint_path':str(ENDPOINT_PATH),'primary_summary_path':str(PRIMARY_PATH),'family_summary_path':str(FAMILY_PATH),'postprimary_validation_one_shot_path':str(VAL_ONE_PATH),'regime_path':str(REGIME_PATH),'large_artifacts_persisted_to_drive':bool(PERSIST_LARGE_ARTIFACTS),'scientific_claim_scope':'Independent corpus+encoder severity-matched replication of the representation-versus-search-effort H3abs ordering under the frozen anchored feedback design. Not a universal mechanism law.'}; REPORT_PATH=OUT/'v027_final_report.json'; REPORT_PATH.write_text(json.dumps(REPORT,indent=2,sort_keys=True)); artifact_paths=[CALIBRATION_PROTOCOL_PATH,CONFIRM_PROTOCOL_PATH,SPLIT_PATH,POLICY_PATH,FIT_ONE_SHOT_PATH,MATCH_TABLE_PATH,TRAJ_PATH,ENDPOINT_PATH,PRIMARY_PATH,FAMILY_PATH,GATE_PATH,VAL_ONE_PATH,REGIME_PATH,REPORT_PATH]; hash_df=pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in artifact_paths]); HASH_PATH=OUT/'V027_ARTIFACT_SHA256.csv'; hash_df.to_csv(HASH_PATH,index=False); print('ARC-v0.27 COMPLETE'); print('OUT:',OUT); print('primary supported:',PRIMARY_SUPPORTED); display(hash_df)

## What to send back

After completion, send either the executed notebook or:

- `v027_fit_severity_match_table.csv`
- `v027_frozen_confirmation_protocol.json`
- `v027_primary_paired_query_bootstrap.csv`
- `v027_secondary_family_robustness.csv`
- `v027_primary_confirmation_gate.json`
- `v027_postprimary_validation_one_shot_multimetric.csv`
- `v027_regime_summary.csv`
- `v027_final_report.json`
- `V027_ARTIFACT_SHA256.csv`

The most important confirmatory result is `representation_minus_search_H3abs` and its paired query-bootstrap 95% CI. Do not retune after validation outcomes are observed.